In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from pathlib import Path

from config import YELLOW_PARQUET, HVFHV_PARQUET, YELLOW_CLEAN05_PARQUET, HVFHV_CLEAN05_PARQUET, HVFHV_CLEAN005_PARQUET, SUBWAY_CSV_1, SUBWAY_CSV_2, SUBWAY_PARQUET, NYC_WEATHER_PARQUET, NYC_BOROUGH_MAPPING, HVFHV_CLEAN05_CSV, HVFHV_CLEAN005_CSV, YELLOW_CLEAN05_CSV

In [3]:
zone_lookup = pd.read_csv(NYC_BOROUGH_MAPPING)
weather = pd.read_excel(NYC_WEATHER_PARQUET)

## Clean subway data

In [ ]:
subway1 = pd.read_csv(SUBWAY_CSV_1)
subway2 = pd.read_csv(SUBWAY_CSV_2)
subway = pd.concat([subway1, subway2], ignore_index=True)

cols_to_drop = [
    "transit_mode",
    "station_complex_id",
    "station_complex",
    "transfers",
    "payment_method",
    "fare_class_category",
    "Georeference",
    "latitude",
    "longitude"
]

subway = subway.drop(columns=[c for c in cols_to_drop if c in subway.columns])

subway["datetime"] = pd.to_datetime(subway["transit_timestamp"])
subway["datetime_hour"] = subway["datetime"].dt.floor("h")

# drop rows with missing critical fields
subway = subway.dropna(subset=["datetime_hour", "borough", "ridership"])

# ensure ridership numeric
subway["ridership"] = pd.to_numeric(subway["ridership"], errors="coerce")
subway = subway.dropna(subset=["ridership"])

# aggregate to (hour, borough)
subway_agg = (
    subway.groupby(["datetime_hour", "borough"], as_index=False)
      .agg({
          "ridership": "sum"
      })
      .rename(columns={"ridership": "subway_ridership"})
)
subway_agg.to_parquet(SUBWAY_PARQUET, index=False)
print(f"Subway dataset saved to: {SUBWAY_PARQUET}")

In [ ]:
subway

In [ ]:
subway_agg

## Clean uber/lyft and yellow taxi data

In [3]:
## Read yellow_taxi
yellow = pq.read_table(YELLOW_PARQUET).to_pandas()
print("Yellow shape:", yellow.shape)

Yellow shape: (48722578, 20)


In [7]:
## Read hvfhv data
hvfhv = pq.read_table(HVFHV_PARQUET).to_pandas()
print("hvfhv shape:", hvfhv.shape)

hvfhv shape: (12179484, 25)


In [ ]:
## Given the speed of analysis, further taking 5% of yellow (i.e., now we take 5% of yellow)
yellow = yellow.sample(frac=0.05, random_state=42)
print("Yellow shape:", yellow.shape)

In [ ]:
# ## Given the speed of analysis, further taking 10% of 5% sample of hvfhv (i.e., now we take 0.5% of hvfhv)
# hvfhv = hvfhv.sample(frac=0.1, random_state=42)
# print("hvfhv shape:", hvfhv.shape)

hvfhv shape: (1217948, 25)


### Standardize columns

In [8]:
## Map uber and lyft
hvfhv = hvfhv[hvfhv["hvfhs_license_num"].isin(["HV0003", "HV0005"])]
mapping = {"HV0003": "uber", "HV0005": "lyft"}
hvfhv['hvfhs_license_num'] = hvfhv['hvfhs_license_num'].replace(mapping)
hvfhv = hvfhv.rename(columns={'hvfhs_license_num': 'source'})
hvfhv["source"].value_counts(normalize=True)

# Clean column names and create new columns for hvfhv
hvfhv["pickup_datetime"] = pd.to_datetime(hvfhv["pickup_datetime"])
hvfhv["dropoff_datetime"] = pd.to_datetime(hvfhv["dropoff_datetime"])
hvfhv["trip_duration_min"] = (hvfhv["dropoff_datetime"] - hvfhv["pickup_datetime"]).dt.total_seconds() / 60

hvfhv = hvfhv.rename(columns={'base_passenger_fare': 'fare'})
hvfhv = hvfhv.drop(columns=['dispatching_base_num', 'originating_base_num', 'on_scene_datetime', 'access_a_ride_flag', 'wav_match_flag'])
hvfhv["fare_per_mile"] =  hvfhv["fare"] / hvfhv["trip_miles"]
hvfhv["fare_per_min"] =  hvfhv["fare"] / hvfhv["trip_duration_min"]
hvfhv["take_rate"] = 1-hvfhv["driver_pay"] / hvfhv["fare"]
hvfhv["wait_time"] = (hvfhv["pickup_datetime"] - hvfhv["request_datetime"]).dt.total_seconds() / 60

In [9]:
# Clean column names and create new columns for yellow
yellow["pickup_datetime"] = pd.to_datetime(yellow["tpep_pickup_datetime"])
yellow["dropoff_datetime"] = pd.to_datetime(yellow["tpep_dropoff_datetime"])
yellow["trip_duration_min"] = (yellow["dropoff_datetime"] - yellow["pickup_datetime"]).dt.total_seconds() / 60
yellow = yellow.rename(columns={'trip_distance': 'trip_miles',
                              'fare_amount': 'fare',
                              'tip_amount': 'tips',
                              'tolls_amount': 'tolls'})
yellow["fare_per_mile"] =  yellow["fare"] / yellow["trip_miles"]
yellow["fare_per_min"] =  yellow["fare"] / yellow["trip_duration_min"]
yellow["source"] = "Yellow Taxi"
yellow = yellow.drop(columns=['payment_type', 'passenger_count'])

In [9]:
# Add time features (date, month, hour, day, weekday vs weekend, etc.)
def add_time_features(df):
    df["PU_date"] = df["pickup_datetime"].dt.date
    df["PU_month"] = df["pickup_datetime"].dt.to_period("M").astype(str)
    df["PU_hour"] = df["pickup_datetime"].dt.hour
    df["PU_datetime_hour"] = df["pickup_datetime"].dt.floor("h")
    df["PU_hour"] = df["pickup_datetime"].dt.hour
    df["PU_day_name"] = df["pickup_datetime"].dt.day_name()
    df["PU_weekday_num"] = df["pickup_datetime"].dt.dayofweek
    df["PU_week_category"] = np.where(df["PU_weekday_num"].isin([5, 6]), "Weekend", "Weekday")
    
    df["PU_time_zone"] = pd.cut(
        df["PU_hour"],
        bins=[0, 6, 12, 18, 24],
        labels=["Late Night", "Morning", "Afternoon", "Evening"],
        right=False
    )
    return df

hvfhv = add_time_features(hvfhv)

# Add peak hour flag
hvfhv["PU_peak_flag"] = hvfhv["PU_hour"].isin([7,8,9,17,18,19]).astype(int)

In [11]:
yellow = add_time_features(yellow)

# Add peak hour flag
yellow["PU_peak_flag"] = yellow["PU_hour"].isin([7,8,9,17,18,19]).astype(int)

### Clean obvious outliers

In [10]:
def clean_hvfhv_trips(df):
    return df[
        (df["trip_miles"] > 0) &
        (df["trip_miles"] < 100) &
        (df["trip_duration_min"] > 0) &
        (df["trip_duration_min"] < 3000) &
        (df["take_rate"] >= -1) &
        (df["take_rate"] < 1) &
        (df["fare"] >= 0) &
        (df["fare"] < 500) &
        (df["tips"] >= 0) &
        (df["tips"] < 500) &
        (df["tolls"] >= 0) &
        (df["tolls"] < 500) &
        (df["sales_tax"] >= 0) &
        (df["sales_tax"] < 500) &
        (df["congestion_surcharge"] >= 0) &
        (df["congestion_surcharge"] < 500) &
        (df["fare"] >= 0) &
        (df["fare"] < 500) &
        (df["fare_per_mile"] > 0) &
        (df["fare_per_mile"] < 100) &
        (df["fare_per_min"] > 0) &
        (df["fare_per_min"] < 100) &
        (df["wait_time"] >= 0) &
        (df["wait_time"] < 200) &
        (df["cbd_congestion_fee"] >= 0) &
        (df["cbd_congestion_fee"] < 500) &
        (df["driver_pay"] > 0) &
        (df["driver_pay"] < 500) &
        (df["airport_fee"] >= 0) &
        (df["airport_fee"] < 500)
    ].copy()


hvfhv = clean_hvfhv_trips(hvfhv)

In [13]:
def clean_yellow_trips(df):
    return df[
        (df["trip_miles"] > 0) &
        (df["trip_miles"] < 100) &
        (df["trip_duration_min"] > 0) &
        (df["trip_duration_min"] < 240) &
        (df["fare"] >= 0) &
        (df["fare"] < 500) &
        (df["fare_per_mile"] > 0) &
        (df["fare_per_mile"] < 100) &
        (df["fare_per_min"] > 0) &
        (df["fare_per_min"] < 100) &
        (df["tips"] >= 0) &
        (df["tips"] < 500) &
        (df["total_amount"] > 0) &
        (df["total_amount"] < 500) &
        (df["tolls"] >= 0) &
        (df["tolls"] < 500) &
        (df["mta_tax"] >= 0) &
        (df["mta_tax"] < 500) &
        (df["congestion_surcharge"] >= 0) &
        (df["congestion_surcharge"] < 500) &
        (df["cbd_congestion_fee"] >= 0) &
        (df["cbd_congestion_fee"] < 500)
    ].copy()

yellow = clean_yellow_trips(yellow)


### Data quality & Summary Stats

In [11]:
def missing_summary(df, name):
    out = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": df.isna().mean().values
    })
    out["dataset"] = name
    return out.sort_values("missing_pct", ascending=False)


missing_hvfhv = missing_summary(hvfhv, "hvfhv Uber/Lyft")
display(missing_hvfhv)

key_cols_hvfhv = ["trip_miles", "trip_duration_min", "fare", "driver_pay", "fare_per_mile", "fare_per_min", "take_rate", "wait_time"]
display(hvfhv[key_cols_hvfhv].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

,column,missing_count,missing_pct,dataset
0,source,0,0.0,hvfhv Uber/Lyft
1,request_datetime,0,0.0,hvfhv Uber/Lyft
2,pickup_datetime,0,0.0,hvfhv Uber/Lyft
3,dropoff_datetime,0,0.0,hvfhv Uber/Lyft
4,PULocationID,0,0.0,hvfhv Uber/Lyft
5,DOLocationID,0,0.0,hvfhv Uber/Lyft
6,trip_miles,0,0.0,hvfhv Uber/Lyft
7,trip_time,0,0.0,hvfhv Uber/Lyft
8,fare,0,0.0,hvfhv Uber/Lyft
9,tolls,0,0.0,hvfhv Uber/Lyft


,trip_miles,trip_duration_min,fare,driver_pay,fare_per_mile,fare_per_min,take_rate,wait_time
count,1.199164e+07,1.199164e+07,1.199164e+07,1.199164e+07,1.199164e+07,1.199164e+07,1.199164e+07,1.199164e+07
mean,4.967089e+00,1.973251e+01,2.673728e+01,2.058119e+01,7.626458e+00,1.409489e+00,2.159897e-01,4.809296e+00
std,5.713905e+00,1.412349e+01,2.402286e+01,1.766973e+01,5.147452e+00,6.191677e-01,2.176986e-01,3.168659e+00
min,2.000000e-02,2.000000e-01,3.300000e-01,1.000000e-02,3.288824e-01,4.925373e-02,-1.000000e+00,0.000000e+00
1%,4.700000e-01,3.216667e+00,6.180000e+00,4.000000e+00,2.273103e+00,5.775852e-01,-5.156873e-01,9.666667e-01
5%,7.500000e-01,5.050000e+00,7.830000e+00,4.820000e+00,3.029820e+00,7.625889e-01,-1.828326e-01,1.550000e+00
25%,1.530000e+00,9.866667e+00,1.249000e+01,9.100000e+00,4.627451e+00,1.045798e+00,1.058023e-01,2.800000e+00
50%,2.960000e+00,1.603333e+01,1.948000e+01,1.540000e+01,6.433121e+00,1.277419e+00,2.458432e-01,4.033333e+00
75%,6.222000e+00,2.541667e+01,3.195000e+01,2.598000e+01,8.909492e+00,1.597625e+00,3.701431e-01,5.866667e+00
95%,1.617000e+01,4.708333e+01,7.048000e+01,5.369000e+01,1.615804e+01,2.496955e+00,4.978541e-01,1.073333e+01


In [ ]:
missing_yellow = missing_summary(yellow, "Yellow Taxi")
display(missing_yellow)
key_cols_yellow = ["trip_miles", "trip_duration_min", "fare", "total_amount", "fare_per_mile", "fare_per_min"]
display(yellow[key_cols_yellow].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

### Define borough

In [12]:
# Define pick up borough 
hvfhv = hvfhv.merge(
    zone_lookup.rename(columns={
        "LocationID": "PULocationID",
        "Borough": "PU_Borough"
    }),
    on="PULocationID",
    how="left"
)

# drop rows with missing critical fields
hvfhv = hvfhv.dropna(subset=["PU_Borough"])


In [16]:
# Define pick up borough 
yellow = yellow.merge(
    zone_lookup.rename(columns={
        "LocationID": "PULocationID",
        "Borough": "PU_Borough"
    }),
    on="PULocationID",
    how="left"
)

# drop rows with missing critical fields
yellow = yellow.dropna(subset=["PU_Borough"])

In [13]:
# Create airport dummy
hvfhv["airport_dummy"] = (
    hvfhv["service_zone"] == "Airports"
).astype(int)
hvfhv = hvfhv.drop(columns=["service_zone"])
print(hvfhv["airport_dummy"].value_counts())

airport_dummy
0    11543574
1      447560
Name: count, dtype: int64


In [18]:
# Create airport dummy
yellow["airport_dummy"] = (
    yellow["service_zone"] == "Airports"
).astype(int)
yellow = yellow.drop(columns=["service_zone"])
print(yellow["airport_dummy"].value_counts())

airport_dummy
0    1605229
1     152623
Name: count, dtype: int64


### Merge subway rideship # by borough to hvfhv

In [14]:
subway = pq.read_table(SUBWAY_PARQUET).to_pandas()

In [15]:
subway

,datetime_hour,borough,subway_ridership
0,2025-01-01 03:00:00,Bronx,760.0
1,2025-01-01 03:00:00,Brooklyn,4659.0
2,2025-01-01 03:00:00,Manhattan,11242.0
3,2025-01-01 03:00:00,Queens,2007.0
4,2025-01-01 04:00:00,Bronx,1364.0
...,...,...,...
35011,2025-12-31 22:00:00,Queens,11207.0
35012,2025-12-31 23:00:00,Bronx,1994.0
35013,2025-12-31 23:00:00,Brooklyn,10263.0
35014,2025-12-31 23:00:00,Manhattan,27268.0


In [16]:
hvfhv = hvfhv.merge(
    subway,
    left_on=["PU_datetime_hour", "PU_Borough"],
    right_on=["datetime_hour", "borough"],
    how="left"
)
hvfhv = hvfhv.drop(columns=["borough", "datetime_hour"])

### Merge weather to hvfhv

In [17]:
weather = weather.replace("T", 0.001)

In [18]:
hvfhv["PU_date"] = pd.to_datetime(hvfhv["PU_date"]).dt.date
weather["date"] = pd.to_datetime(weather["date"]).dt.date

In [19]:
hvfhv = hvfhv.merge(
    weather[["date", "tmax_f", "tmin_f", "rain_melted_snow_etc_in"]],
    left_on=["PU_date"],
    right_on=["date"],
    how="left"
)

hvfhv = hvfhv.drop(columns=["date"])

In [20]:
hvfhv

,source,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,fare,tolls,...,PU_week_category,PU_time_zone,PU_peak_flag,PU_Borough,Zone,airport_dummy,subway_ridership,tmax_f,tmin_f,rain_melted_snow_etc_in
0,uber,2025-01-05 11:33:26,2025-01-05 11:34:58,2025-01-05 11:54:18,163,144,3.450,1160,27.20,0.00,...,Weekend,Morning,0,Manhattan,Midtown North,0,47628.0,33,28,0
1,uber,2025-01-16 22:52:08,2025-01-16 22:55:59,2025-01-16 23:33:40,155,265,19.010,2261,53.35,0.00,...,Weekday,Evening,0,Brooklyn,Marine Park/Mill Basin,0,13043.0,30,23,0.001
2,uber,2025-01-19 23:09:59,2025-01-19 23:28:09,2025-01-19 23:41:19,106,14,5.950,790,18.74,0.00,...,Weekend,Evening,0,Brooklyn,Gowanus,0,6853.0,41,24,0.26
3,lyft,2025-01-23 10:50:08,2025-01-23 10:53:50,2025-01-23 11:10:11,180,95,3.467,981,17.85,0.00,...,Weekday,Morning,0,Queens,Ozone Park,0,28182.0,28,17,0
4,lyft,2025-01-19 01:35:56,2025-01-19 01:44:22,2025-01-19 01:47:21,91,72,0.508,179,8.06,0.00,...,Weekend,Late Night,0,Brooklyn,Flatlands,0,3750.0,41,24,0.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11991129,uber,2025-12-19 08:15:55,2025-12-19 08:19:08,2025-12-19 08:24:13,244,244,0.690,305,22.40,0.00,...,Weekday,Morning,1,Manhattan,Washington Heights South,0,63205.0,58,32,1.16
11991130,lyft,2025-12-24 15:50:27,2025-12-24 15:54:28,2025-12-24 16:05:37,113,144,1.248,669,12.27,0.00,...,Weekday,Afternoon,0,Manhattan,Greenwich Village North,0,60588.0,46,35,0
11991131,uber,2025-12-03 20:35:58,2025-12-03 20:45:30,2025-12-03 21:00:44,186,162,1.600,914,20.86,0.00,...,Weekday,Evening,0,Manhattan,Penn Station/Madison Sq West,0,52664.0,41,31,0
11991132,uber,2025-12-18 18:39:28,2025-12-18 18:41:48,2025-12-18 19:02:30,167,182,4.630,1242,20.33,0.00,...,Weekday,Evening,1,Bronx,Morrisania/Melrose,0,11274.0,50,36,0.02


### Export the clean parquet files

In [ ]:
yellow.to_parquet(
    YELLOW_CLEAN05_PARQUET,
    index=False,
    engine="pyarrow"
)


In [ ]:
hvfhv.to_parquet(
    HVFHV_CLEAN05_PARQUET, # HVFHV_CLEAN005_PARQUET,
    index=False,
    engine="pyarrow"
)

### Optional: Export the clean csv files (takes longer)

In [ ]:
def parquet_to_csv_stream(parquet_path, output_path):
    pf = pq.ParquetFile(parquet_path)

    output_path = Path(output_path)
    if output_path.exists():
        output_path.unlink()

    first_chunk = True

    for i in range(pf.num_row_groups):
        table = pf.read_row_group(i)
        df = table.to_pandas()

        # Write header only once
        df.to_csv(
            output_path,
            mode="a",
            index=False,
            header=first_chunk
        )

        first_chunk = False

    print(f"Saved: {output_path}")  

In [ ]:
parquet_to_csv_stream(
    YELLOW_CLEAN05_PARQUET,
    YELLOW_CLEAN05_CSV
)

parquet_to_csv_stream(
    HVFHV_CLEAN005_PARQUET,
    HVFHV_CLEAN005_CSV,
)

Saved: c:\Users\kathy.zhang\OneDrive - MMC\Documents\Project Setup\Uber\uber-analytics\uber_analytics\data\processed\yellow_taxi\yellow_taxi_2025_clean05.csv
Saved: c:\Users\kathy.zhang\OneDrive - MMC\Documents\Project Setup\Uber\uber-analytics\uber_analytics\data\processed\hvfhv\hvfhv_2025_uber_lyft_clean005.csv
